# Tutorial 6 · Deep learning on recordings

AI in Rehabilitation and Healthcare (376-1229-00L) · HS 2026 · Deep learning for physiological signals

**What you'll do:**
1. Turn the feature table from Tutorial 2 into **one sequence per recording**: a person doing one trial, its 10-second windows in time order, the features as channels.
2. Split **by person**, so no one is on both sides.
3. Train a **shallow baseline** (a random forest on each recording's average features) and two networks from `sktime`: a **multilayer perceptron (MLP)** and an **LSTM-FCN**.
4. Compare them on the **same test people**, and set the number of weights against the number of people they learn from.

**What one sample is.** A recording here is a few hundred numbers per window, not raw signal: about 23 windows by 60 features, and there are only a few hundred recordings from 24 people. A network has to earn its place against a forest on the same people; that comparison is the point of this tutorial.

**With the course coach.** Run *T6.00* with your code from SCAI Learning (*Your data → Connect a notebook*). The coach then sees the code of each cell you run, and the *Report* cells send it numbers about your split, your scores and your model, never your data and never a cell's output. Ask it about a cell by its name, for example *"why does T6.05 split by person?"*. Without a code the notebook runs the same and sends nothing.


In [ ]:
# T6.00 Connect the course coach (optional)
# In SCAI Learning, open Your data → Connect a notebook and copy the six-letter code. Paste it
# between the quotes and run this cell. Left empty, nothing is installed and nothing is sent.
# Connected, the coach receives the code of each cell you run and the numbers the Report cells
# compute: never your data, never a cell's output.
COACH_CODE = ""
coach = globals().get("coach")          # connected already with the app's snippet? keep it
if COACH_CODE and coach is None:
    %pip install -q https://app.scai-learning.ch/companion.tar.gz
    import rehab_coach as coach
    coach.connect(COACH_CODE)
    coach.watch()
print("Coach connected: the cells you run from here on reach it." if coach else "Working without the coach.")


## 1) Setup


In [ ]:
# T6.01 Install libraries
# sktime's networks need TensorFlow. Colab may have it already; this installs what is missing.
%pip install -q sktime
import importlib.util
if importlib.util.find_spec("tensorflow") is None:
    %pip install -q tensorflow


In [ ]:
# T6.02 Mount Google Drive
# Colab only. On Jupyter on your own computer, skip this cell and set data_path below to your folder.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# T6.03 Load the feature table
import numpy as np
import pandas as pd
from pathlib import Path

data_path = Path('/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/')   # Jupyter on your own computer: the folder that holds the file
df = pd.read_csv(data_path / 'features_df_10s_3s.csv')
print(df.shape, '·', df['subject'].nunique(), 'people ·', df.groupby(['subject', 'trial']).ngroups, 'recordings')


## 2) One sequence per recording

Each recording (one person, one trial) becomes a 2-D array: **channels** (the 60 features) by **time** (its windows, in order). Recordings differ in length, so each is trimmed or padded to the average length by repeating its last window. The `window` column is the window's position, not a measurement, so it is not a channel: kept in, padding would let it encode how long a recording is.


In [ ]:
# T6.04 Build one sequence per recording
feature_cols = [c for c in df.columns if c not in ('subject', 'activity', 'trial', 'window')]
recordings = df.sort_values('window').groupby(['subject', 'trial'], sort=True)
target_length = int(round(recordings.size().mean()))
print('Target length (windows):', target_length)

samples, labels, people = [], [], []
for (subject, trial), rec in recordings:
    a = rec[feature_cols].to_numpy().T                       # (channels, windows)
    if a.shape[1] >= target_length:
        a = a[:, :target_length]                             # trim
    else:
        a = np.concatenate([a, np.repeat(a[:, -1:], target_length - a.shape[1], axis=1)], axis=1)   # pad with the last window
    samples.append(a); labels.append(rec['activity'].iloc[0]); people.append(subject)

X_3d = np.array(samples)           # (recordings, channels, time)
y_seq = np.array(labels)
subjects_seq = np.array(people)
print('X_3d shape:', X_3d.shape, '·', len(np.unique(subjects_seq)), 'people')


## 3) Split by person

In 2025 this cell used `train_test_split` on recordings: almost every person ended up on both sides, and the scores changed on every run. A person's recordings must stay together.


In [ ]:
# T6.05 Split by person
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X_3d, y_seq, groups=subjects_seq))
X_train, X_test = X_3d[train_idx], X_3d[test_idx]
y_train, y_test = y_seq[train_idx], y_seq[test_idx]
print(f'Train: {len(train_idx)} recordings from {len(np.unique(subjects_seq[train_idx]))} people')
print(f'Test:  {len(test_idx)} recordings from {len(np.unique(subjects_seq[test_idx]))} people')
print('People on both sides:', len(set(subjects_seq[train_idx]) & set(subjects_seq[test_idx])))


Networks learn badly from inputs on very different scales: here a mean acceleration sits near 0 and a dominant power in the thousands. Scale each feature with the mean and standard deviation of the **training people** only; the test people must not shape the scaling either.


In [ ]:
# T6.06 Scale each feature on the training people
mu = X_train.mean(axis=(0, 2), keepdims=True)          # one mean and one SD per feature (channel)
sd = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
X_train_s, X_test_s = (X_train - mu) / sd, (X_test - mu) / sd
print('Scaled with', X_train.shape[0], 'training recordings; test recordings used:', 0)


In [ ]:
# T6.07 Report the split
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T6.00).
if coach:
    coach.baseline(y_seq)                       # the zero-rate score: always the commonest activity
    coach.report((train_idx, test_idx), subjects_seq, y_seq, tutorial="T6")
else:
    print("Not connected: nothing sent. Run T6.00 with your code to use the coach.")


## 4) A shallow baseline first

A random forest on each recording's **average** features, trained and tested on exactly the same people. Whatever a network scores below is read against this number, and against the zero-rate score (always predicting the commonest activity).


In [ ]:
# T6.08 Shallow baseline: a forest on average features
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix

def summarise(X):                     # (recordings, channels, time) -> (recordings, channels)
    return X.mean(axis=2)

forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
forest.fit(summarise(X_train), y_train)
y_pred_forest = forest.predict(summarise(X_test))

values, counts = np.unique(y_train, return_counts=True)
zero_rate = balanced_accuracy_score(y_test, np.full(len(y_test), values[counts.argmax()]))
results = {'zero-rate': zero_rate, 'random forest': balanced_accuracy_score(y_test, y_pred_forest)}
print(f"Zero-rate balanced accuracy: {zero_rate:.3f}")
print(f"Forest balanced accuracy:    {results['random forest']:.3f}")


## 5) A multilayer perceptron


In [ ]:
# T6.09 Train the MLP
from sktime.classification.deep_learning.mlp import MLPClassifier

mlp = MLPClassifier(n_epochs=100, batch_size=64, random_state=42, verbose=False)
mlp.fit(X_train_s, y_train)
n_weights = mlp.model_.count_params()
n_people = len(np.unique(subjects_seq[train_idx]))
print(f'MLP weights: {n_weights:,} · training people: {n_people} · {n_weights // n_people:,} weights per person')


In [ ]:
# T6.10 Score the MLP
import matplotlib.pyplot as plt
import seaborn as sns

def show(name, y_pred):
    results[name] = balanced_accuracy_score(y_test, y_pred)
    print(f'{name}: accuracy {accuracy_score(y_test, y_pred):.3f} · balanced accuracy {results[name]:.3f} · '
          f'F1-macro {f1_score(y_test, y_pred, average="macro"):.3f}')
    print(classification_report(y_test, y_pred, zero_division=0))
    classes = np.unique(y_seq)
    plt.figure(figsize=(7, 5))
    sns.heatmap(confusion_matrix(y_test, y_pred, labels=classes), annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(f'{name} · test people'); plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()

show('MLP', mlp.predict(X_test_s))


## 6) An LSTM-FCN

A convolutional branch and a recurrent branch side by side (Karim et al., 2018). Convolutions share their weights along time, so a pattern is recognised wherever it occurs; the MLP gives every time step its own weights.


In [ ]:
# T6.11 Train the LSTM-FCN
from sktime.classification.deep_learning.lstmfcn import LSTMFCNClassifier

lstmfcn = LSTMFCNClassifier(n_epochs=100, batch_size=64, random_state=42, verbose=False)
lstmfcn.fit(X_train_s, y_train)
print(f'LSTM-FCN weights: {lstmfcn.model_.count_params():,}')


In [ ]:
# T6.12 Score the LSTM-FCN
show('LSTM-FCN', lstmfcn.predict(X_test_s))


## 7) The comparison

One table, the same test people for every row. Before calling a network better, ask how many test people stand behind the difference: one person more or less right moves these numbers by several points.


In [ ]:
# T6.13 Compare on the same people
best_deep = max(('MLP', 'LSTM-FCN'), key=lambda k: results[k])
gap = results[best_deep] - results['random forest']
print(pd.Series(results, name='balanced accuracy').round(3).to_string())
print(f"\n{best_deep} minus the forest: {gap:+.3f}, on {len(np.unique(subjects_seq[test_idx]))} test people")


In [ ]:
# T6.14 Report the comparison
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T6.00).
if coach:
    coach.report((train_idx, test_idx), subjects_seq, y_seq, model=lstmfcn, tutorial="T6",
                 scores={"balanced_accuracy_forest": results['random forest'], "balanced_accuracy_mlp": results['MLP'],
                         "balanced_accuracy_lstmfcn": results['LSTM-FCN']},
                 lesson={"shallow_vs_deep_gap": gap})
else:
    print("Not connected: nothing sent. Run T6.00 with your code to use the coach.")


## ✍️ Exercise

1. **Another architecture, same comparison.** Train `MACNNClassifier` from `sktime` ([documentation](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.classification.deep_learning.macnn.MACNNClassifier.html)) on the same split and add it to the table, beside the forest.
2. **Folds, not one split.** Five test people is a small jury. Repeat the comparison over `GroupKFold(n_splits=5)` on `subjects_seq`: per fold, fit the forest and your best network on the same training people and score both on the same test people. Report the mean, the spread and the per-fold difference.
3. **Read the difference.** On how many folds does the network beat the forest, and by more than the spread between folds?
